# Lesson 01c — Prompt Engineering Deep Dive
**Domain:** E-commerce · Amazon Reviews  
**Dataset:** `McAuley-Lab/Amazon-Reviews-2023` (Electronics, HuggingFace)  
**Time estimate:** ~2 h

## Learning objectives
- Zero-shot, few-shot, chain-of-thought (CoT), self-consistency
- XML tags as structure anchors (Anthropic pattern)
- Role prompting and persona definition
- Iterative refinement loop: prompt → test → measure → improve

In [ ]:
import sys, re
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from openai import OpenAI
from llm_checker import check, hint, evaluate, progress

local_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
MODEL = "local-model"

def llm(messages, max_tokens=60, temperature=0.0):
    resp = local_client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=temperature
    )
    return resp.choices[0].message.content.strip()

print("✅ Setup complete")

## Load Amazon Reviews

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full",
    trust_remote_code=True,
).select(range(500))

# Build 30 reviews with manual labels for accuracy testing
CATEGORIES = ["Positive", "Negative", "Feature Request", "Shipping Issue", "Other"]

reviews_30 = [ds[i]["text"][:300] for i in range(30)]

# Manual labels (assign based on a quick read — adapt these to your actual data)
# For the exercise, we create approximate labels; in production you'd label manually
import random; random.seed(42)
true_labels = [random.choice(CATEGORIES) for _ in range(30)]   # placeholder labels

print(f"Loaded {len(reviews_30)} reviews for classification exercises")

---
## 🔵 EXAMPLE — Ex 01c-1: Few-shot vs zero-shot accuracy

Classify 30 Amazon reviews into 5 categories. Compare zero-shot vs 3-shot few-shot accuracy.

In [ ]:
# ── Ex 01c-1: Few-shot vs zero-shot (EXAMPLE) ─────────────────────

SYSTEM = "You are a product review classifier. Respond with exactly one category name."
LABEL_LIST = ", ".join(CATEGORIES)

# ── Zero-shot prompt ──────────────────────────────────────────────
def classify_zero_shot(review: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": f"Categories: {LABEL_LIST}\n\nReview: {review}\n\nCategory:"},
    ]
    return llm(messages)

# ── Few-shot prompt (3 hand-crafted examples) ─────────────────────
FEW_SHOT_EXAMPLES = [
    ("Great battery life, charges fast, highly recommend!", "Positive"),
    ("Stopped working after 2 weeks, total waste of money.", "Negative"),
    ("Wish it had Bluetooth — please add in next version!",   "Feature Request"),
]

def classify_few_shot(review: str) -> str:
    messages = [{"role": "system", "content": SYSTEM}]
    for ex_review, ex_label in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user",      "content": f"Review: {ex_review}\nCategory:"})
        messages.append({"role": "assistant", "content": ex_label})
    messages.append({"role": "user", "content": f"Categories: {LABEL_LIST}\n\nReview: {review}\nCategory:"})
    return llm(messages)

# Run on 30 reviews
zs_preds = [classify_zero_shot(r) for r in reviews_30]
fs_preds = [classify_few_shot(r)  for r in reviews_30]

def accuracy(preds, labels):
    return sum(p.strip().lower() == l.strip().lower() for p, l in zip(preds, labels)) / len(labels)

zs_acc = accuracy(zs_preds, true_labels)
fs_acc = accuracy(fs_preds, true_labels)

print(f"Zero-shot accuracy : {zs_acc:.0%}")
print(f"Few-shot  accuracy : {fs_acc:.0%}")
print(f"Did few-shot help? : {'Yes ✅' if fs_acc >= zs_acc else 'No — try different examples'}")

check([
    (isinstance(zs_acc, float), "Zero-shot accuracy printed as float"),
    (isinstance(fs_acc, float), "Few-shot accuracy printed as float"),
    (0 <= zs_acc <= 1,          "Accuracy in [0, 1]"),
], exercise_id="01c-1")

---
## 🟡 EXERCISE — Ex 01c-2: CoT with XML extraction

Add to your prompt: `"Think step by step. First write <reasoning>...</reasoning>,
then write <category>...</category>."`  
Parse both fields from the response using `re`. Run on 30 reviews.

**Auto-check verifies:**
- XML parsing succeeds for ≥ 90% of responses
- `category` extracted correctly from tag
- `reasoning` is non-empty (≥ 10 words)

In [ ]:
def classify_cot_xml(review: str) -> dict:
    """
    Returns {'reasoning': str, 'category': str} parsed from XML tags.
    Returns {'reasoning': '', 'category': ''} if parsing fails.
    """
    # TODO: implement
    # 1. Build a prompt instructing the LLM to output <reasoning>...</reasoning>
    #    followed by <category>...</category>
    # 2. Call llm(messages, max_tokens=200)
    # 3. Parse with re.search(r'<reasoning>(.*?)</reasoning>', response, re.DOTALL)
    raise NotImplementedError("Implement classify_cot_xml()")

In [ ]:
# ── Auto-check ───────────────────────────────────────────────────────
try:
    cot_results = [classify_cot_xml(r) for r in reviews_30]

    parse_ok    = sum(1 for r in cot_results if r["category"] != "")
    reasoning_ok = sum(1 for r in cot_results if len(r["reasoning"].split()) >= 10)
    parse_rate   = parse_ok / len(reviews_30)

    print(f"Parse success rate : {parse_rate:.0%}  ({parse_ok}/30)")
    print(f"Reasoning ≥10 words: {reasoning_ok}/30")

    check([
        (parse_rate >= 0.9,   f"XML parsing succeeds ≥ 90% ({parse_rate:.0%})"),
        (reasoning_ok >= 24,  "Reasoning ≥ 10 words in most responses"),
        (all("category" in r for r in cot_results), "All results have 'category' key"),
    ], exercise_id="01c-2")

except NotImplementedError:
    print("⚠️  Implement classify_cot_xml() first.")

---
## 🔴 CHALLENGE — Ex 01c-3: Iterative refinement loop

Start with the prompt from Ex 01c-1. Find 5 misclassified reviews.
Modify the prompt (add a clarification, example, or instruction).
Measure accuracy change. Repeat 3 times.

Document each iteration in a table: `{iteration, change_made, accuracy}`.

**Auto-check verifies:**
- Table with 3+ iterations filled
- At least 1 iteration improves accuracy
- Comment on what worked

In [ ]:
# ── Your refinement log ───────────────────────────────────────────
# Fill this in as you work through the iterations

iterations = [
    # {"iter": 1, "change": "baseline zero-shot",                 "accuracy": zs_acc},
    # {"iter": 2, "change": "added instruction: ...",             "accuracy": ???},
    # {"iter": 3, "change": "...",                                "accuracy": ???},
]

# For each iteration:
# 1. Look at misclassified reviews from the previous run
# 2. Decide what to change in the prompt
# 3. Re-run classify_zero_shot (or write a new version)
# 4. Compute accuracy and append to iterations list

In [ ]:
# ── Auto-check ───────────────────────────────────────────────────────
if len(iterations) >= 3:
    accs = [it["accuracy"] for it in iterations]
    any_improvement = any(accs[i+1] > accs[i] for i in range(len(accs)-1))

    check([
        (len(iterations) >= 3,  "≥ 3 iterations recorded"),
        (any_improvement,       "At least 1 iteration improves accuracy"),
        (all("change" in it for it in iterations), "Each iteration documents the change made"),
    ], exercise_id="01c-3")

    print("\nIteration summary:")
    for it in iterations:
        print(f"  Iter {it['iter']}: acc={it['accuracy']:.0%}  change='{it['change']}'")
else:
    print("⚠️  Complete at least 3 refinement iterations before running auto-check.")

In [ ]:
progress()

---
## Readiness checklist before moving to 01d
- [ ] Few-shot prompt gives higher accuracy than zero-shot (or I understand why it doesn't)
- [ ] CoT with XML tags works and reasoning is parsed correctly
- [ ] Iterative refinement loop: ≥ 2 iterations documented with accuracy delta